## 1) Inspiration

As someone who loves watching movies/shows in their free time, I only think about my own experiences/reactions watching these forms of entertainment rather than the countless tactical decisions and the long process prior to cultivating a final product movie/episode, whether it be budgeting decisions for the set/cast or marketing strategies. Therefore, it would be interesting to delve further into the decisions being made behind the scenes of production companies. It is normal to only enjoy the final product, but I am also interested in the the actual processes of developing films and the tactical decisions that come along with filming and marketing.

In a more general sense, the films can be modeled as a high-risk investments, where even high-budget films underperform. Many factors, such as genres, release times, or runtimes can influence revenue and box office performance. Therefore, it is important for production companies to consider all of these factors and be intentional with each decision in order to both minimize making disastrous mistakes and maximize profit. I'm interested in better understanding financial performance insights about what audiences respond to and shifts in the dynamics of the film industry. As someone who watches a lot of films, it intrigues me to investigate how production companies tailor their decisions to the interests of the audience, keeping film enjoyers like me satisfied with the final product.

## 2) Stakeholders

The main stakeholder for this project would be movie studios and production companies. Predicting profits from films would help them decide on what prioritize for each project based off of audience responses in past projects (e.g. genres, effect of runtime), as well as provide insight on how much money to invest and budget for production and marketing purposes. Therefore, within production companies, another stakeholder for this project include marketing and promotion teams. Identifying potentially high return on investment films are helpful in making decisions on allocating promotional resources effectively. These insights also give light into ways to optimize marketing strategies and tailor campaigns based on predicted box office performances and audience response.

## 3) Task and Metrics

I will be implementing a **regression** task on the data. There are many variables in the data, including voting averages/counts, release dates, runtimes, genres, production companies/countries, languages, and more. From the information given in these variables, I will be predicting the profit a movie makes, investigating the relationship between these predictor variables with profit. 

In terms of **evaluation metrics**, I will be prioritizing evaluating my models using **root mean squared error** (RMSE). RMSE squares every error before averaging, so it is penalized more if it underestimates/overestimates blockbuster profits. This metric is especially valuable because it can capture large mistakes that may be vital for making business/financial decisions in a movie production context. For example, it is essential that the model does not make predictions that are way off, as a small error (e.g. 10 million vs. 12 million dollars) is not as critical as a large error (e.g. 20 million vs. 200 million dollars) when allocating resources for film projects. A prediction that is "way off" can lead to poor investment decisions, overspending, or missing potential hits, which RMSE prioritizes to reduce.

However, I still think **mean absolute error** (MAE) is valuable to incorporate in evaluating my model as well. This metric is the average absolute error, treating all errors equally regardless of magnitude and being more robust to outliers. The model may have errors of 3 million, 5 million, 6 million, 8 million, and 300 million. Here, the MAE is about 64 million, while the RMSE comes out to about 134 million dollars. Looking at the data, revenue alone goes all the way to 5 billion dollars, with most movies clustering at significantly lower values than this. This means that a small number of popular, outlier films disproportionately pulls RMSE up, so even if the model generalizes well to 95%+ of the data with modest profits, it may still look terrible if RMSE was the only metric due to how it handles outliers, as it squares errors before averaging. Ultimately, MAE will be useful in providing a ground-level assessment of the model performance that RMSE may obscure when outliers are present, so it will be valuable to consider both RMSE and MAE in this project.

## 4) Data

#### Basic Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

#### Data Identification
The dataset is titled "Full TMDB Movies Dataset 2024 (1M Movies)", which can be found on Kaggle https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies. The dataset is derived from TMDB (The Movie Database), and it provides information about various movies that have been released or is set to release in the future, including information like titles, ratings, release dates, genres, and more.

#### Data Description
This dataset contains **1,375,026 observations** (where each observation is a movie), and 24 variables. **Note**: The original dataset in Kaggle does not have a "profit" variable. I will be adding this manually, therefore having a total of **25 variables**. 

Of the 25 variables, there are:

- **9 numeric variables**:
    - profit (added manually): The result of subtracting the budget of a movie from a movie's revenue values
    - vote_average: Average vote or rating given by viewers
    - vote_count: Total count of votes received for the movie
    - revenue: Total revenue generated by the movie
    - runtime: Duration of the movie in minutes
    - budget: Budget allocated for the movie
    - popularity: A "lifetime" popularity score of the movie based on user engagement attributes, including number of votes/views/favorites/additions to watchlists per day, release dates, total number of votes, and previous days' scores. [1]
    - id: Unique identifier for each movie

- **6 categorical variables**:
    - status: The status of the movie (e.g. Released, Rumored, Post Production, etc.)
    -  adult: Indicates if the movie is suitable only for adult audiences
    -  original_language: Original language in which the movie was produced
    -  genres: List of genres the movie belongs to
    -  production_companies: List of production companies involved in the movie
    -  production_countries: List of countries involved in the movie production
    -  spoken_languages: List of languages spoken in the movie

The **response** variable will be `profit`, and I will be using `popularity`, `genres`, `release_month` (derived from release_date, shown in "Data Cleaning"), `original_language`, `runtime`, and `adult`, as **predictors**.

#### Data Cleaning

In [ ]:
data_full = pd.read_csv('movies.csv')
# Adding profit column
data_full['profit'] = data_full['revenue'] - data_full['budget']

# Filtering out observations with missing values in any of the predictors I want to use: 
data_clean = data_full.loc[~(data_full['popularity'].isna() |
               data_full['genres'].isna() |
               data_full['runtime'].isna() |
               data_full['release_date'].isna() |
               data_full['adult'].isna() |
               data_full['original_language'].isna())]

# Removing columns with revenue and budget as 0
data_clean = data_clean.loc[~((data_clean['revenue'] == 0) & (data_clean['budget'] == 0))]

Before further filtering and cleaning the data, I created a new `profit` column, which was defined as $revenue - budget$. I also filtered out observations where `revenue` and `budget` was 0 dollars. When both of these are 0, the resulting profit is 0 - 0 = 0. The problem is that this zero doesn't mean the film broke even or had no financial activity, but it may have not been recorded in the database, or is not publicly disclosed. Instead of being stored as a null value, it may have been stored as 0 dollars instead, which is why I decided to filter these observations out.

Next, in the original dataset, there were multiple missing values in the variables that I wanted to use as predictors, particularly in  `genres` and `release_date`:

In [ ]:
data_full[['popularity', 'genres', 'runtime', 'original_language', 'release_date', 'adult']].isna().sum()

popularity                0
genres               597001
runtime                   0
original_language         0
release_date         293527
adult                     0
dtype: int64

Therefore, I filtered out all the observations with missing values in these predictors of interest so analyses isn't conducted on data with missing values.

**#1) runtime**: Next, I noticed that some observations had unrealistic runtimes in minutes:

In [ ]:
print('Observation with extremely high runtime:')
display(data_clean.loc[data_clean['runtime'] == data_clean['runtime'].max(), ['title', 'runtime']])

print(' ')

print('Observation(s) with extremely low runtimes:')
display(data_clean.loc[data_clean['runtime'] == data_clean['runtime'].min(), ['title', 'runtime']].head(3)) 

Observation with extremely high runtime:


,title,runtime
964483,The Innocence,1265


 
Observation(s) with extremely low runtimes:


,title,runtime
18758,Vajont,0
23015,Foxter and Max,0
24920,Il regno,0


A movie with a runtime of 1,265 minutes (21 hours) is an extreme outlier that may negatively affect my analyses, and a movie with 0 minutes is simply unrealistic. To navigate extreme runtimes, I created boundaries out of personal judgement to filter out observations. I ultimately decided on keeping observations with a runtime longer than 30 minutes and shorter than 5 hours (300 minutes) to account for the variety of forms of entertainment (short films, documentaries, etc.) while removing any extreme outliers that may have a negative impact on analysis.

In [ ]:
# Observations where runtime is greater than 30 minutes and less than 300 minutes (5 hours)
data_clean = data_clean.loc[(data_clean['runtime'] >= 30) & (data_clean['runtime'] <= 300)]

**#3) genres**:

Additionally, because the `genres` variable lists multiple genres for each movie, there is a lot of unique combinations/categories. For example, "Action, Science Fiction, Adventure" and "Science Fiction, Action, Adventure" are seen as different categories, even though they contain the same elements. To mitigate this issue, I encoded the genres column using MultiLabelBinarizer in scikit-learn, which converts a collection of sets or lists of categorical labels into a binary indicator matrix (i.e one-hot-encodes multi-label data), using commas in the original genres variable to distinguish between distinct genres. For genres that appeared in less than 1% of films, I labeled these rare genres as "Other".

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Encoding the "genres" column so that it is no longer a long, singular list with many unique combinations
mlb_genres = MultiLabelBinarizer()
genres_split = data_clean['genres'].str.split(', ') # using the comma , to split genres
genres_encoded = mlb_genres.fit_transform(genres_split)
genres_df = pd.DataFrame(genres_encoded, columns = mlb_genres.classes_, index = data_clean.index)

# Collapse rare genres into 'Other' (appear in less than 1% of films)
threshold = 0.01
genre_counts = genres_df.sum()
common_genres = genre_counts[genre_counts / len(genres_df) >= threshold].index
rare_genres = genre_counts[genre_counts / len(genres_df) < threshold].index

genres_df['Other_Genre'] = (genres_df[rare_genres].sum(axis = 1) > 0)
genres_df = genres_df[list(common_genres) + ['Other_Genre']]

# Drop original and join encoded genres
data_clean = data_clean.drop(columns = ['genres'])
data_clean = data_clean.join(genres_df)

**#4) original_language**: 

As for `original_language`, there were many distinct languages, many with only 1 observation (movie) in that language. To prevent too many language categories, I collapsed languages that appeared in less than 1% of the observations into the label as "Other". After doing so, the number of unique values in original_language went from 171 to 11.

In [ ]:
# Cleaning original_language variable
min_pct = 0.01
lang_counts = data_clean['original_language'].value_counts()
top_orig_langs = lang_counts[lang_counts / len(data_clean) >= min_pct].index.tolist()

data_clean['original_language'] = data_clean['original_language'].apply(
    lambda x: x if x in top_orig_langs else 'other')

# Table
lang_counts = data_clean['original_language'].value_counts()
lang_pct = (lang_counts / len(data_clean) * 100).round(2)

lang_table = pd.DataFrame({
    'Language': lang_counts.index,
    'Count': lang_counts.values,
    'Percentage': lang_pct.values
}).sort_values('Percentage', ascending = False).reset_index(drop=True)

lang_table['Percentage'] = lang_table['Percentage'].apply(lambda x: f"{x}%")

lang_table.style.hide(axis = 'index')

Language,Count,Percentage
en,23441,69.92%
other,3978,11.87%
fr,1252,3.73%
es,1143,3.41%
ru,687,2.05%
ja,622,1.86%
hi,564,1.68%
de,547,1.63%
zh,477,1.42%
it,459,1.37%


**#5) release_date**:

As for `release_date`, this column is in the format "YEAR - MONTH - DAY". I decided to extract the month from each observation to investigate possible relationships between months/seasons with movie profits further and didn't consider the year and day of release. This was done by converting the release_date column to a datetime object and then extracting the month using the object's respective attribute.

In [ ]:
# Extracting month only from the release_date column
data_clean['release_date'] = pd.to_datetime(data_clean['release_date'])
data_clean['release_month'] = data_clean['release_date'].dt.month

data_clean = data_clean.drop(columns = ['release_date']) # Removing original release_date column

**#6) Random sampling and splitting data**:

Lastly, after these cleaning and filtering steps, **33,526** observations remained. I therefore decided to **randomly sample 25,000** observations using `.sample()`, setting random_state = 16 for reproducibility. I decided to sample a smaller subset of observations due to computational practicality reasons, so that I am still able to receive some insights about the relationship between the predictors without extreme runtimes in the code.

Finally, the dataset that will be used for this regression task will have 25,000 observations and 44 variables (many of the additional variables being distinct genres as its own column rather than collapsed into a singular list). 

I then used `train_test_split` with an 80-20 split between training and test sets and then one-hot-encoded original_language and release_month to prepare the data for the regression task.

In [ ]:
# Randomly sampling (without replacement) 25,000 observations due to computational practicality

data = data_clean.sample(n = 25000, random_state = 16) # random_state for reproducibility

In [ ]:
pd.DataFrame([{'Number of Observations': data.shape[0],
             'Number of Columns': data.shape[1]}]).style.hide(axis = 'index')

Number of Observations,Number of Columns
25000,44


In [ ]:
genre_cols = [col for col in data.columns if col in list(common_genres) + ['Other_Genre']]
feature_cols = ['popularity', 'runtime', 'original_language', 'release_month', 'adult'] + genre_cols

y = data['profit']
X = data[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 16)



# One-hot-encoding original_language and release_month
    # original_language
X_train = pd.get_dummies(X_train, columns = ['original_language'], drop_first = True)
X_test = pd.get_dummies(X_test, columns = ['original_language'], drop_first = True)
X_test = X_test.reindex(columns = X_train.columns) # Align columns in case some categories only appear in one split

    # release_month
X_train = pd.get_dummies(X_train, columns = ['release_month'], drop_first = True)
X_test = pd.get_dummies(X_test, columns = ['release_month'], drop_first = True)
X_test = X_test.reindex(columns = X_train.columns) # Align columns in case some categories only appear in one split

## 5) Prediction

#### Unregularized (Linear) Model

The first predictor I decided to use was `popularity`. The main reason for doing so is because it appears to be an intuitive starting point: it is a continuous numeric variable and about the most direct proxy for how much attention a film attracts and revenue a film generates. After training a linear and unregularized model using this predictor, the training and test performances are as follows:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

model = LinearRegression()
model.fit(X_train[['popularity']], y_train)

y_pred_train = model.predict(X_train[['popularity']])
y_pred_test = model.predict(X_test[['popularity']])

pd.DataFrame([{
    'Predictors': 'popularity',
    'Train RMSE': round(root_mean_squared_error(y_train, y_pred_train), 2),
    'Test RMSE': round(root_mean_squared_error(y_test, y_pred_test), 2),
    'Train MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2),
    'Train R²': round(model.score(X_train[['popularity']], y_train), 4),
    'Test R²': round(model.score(X_test[['popularity']], y_test), 4)}]).style.hide(axis = 'index')

Predictors,Train RMSE,Test RMSE,Train MAE,Test MAE,Train R²,Test R²
popularity,85513064.020000,74852384.120000,24842043.400000,23697003.660000,0.057200,0.005600


After training the model using only popularity, I then added the rest of the predictors along with popularity one at a time, in the following order: `genres`, `release_month`, `runtime`, `original_language`, `adult`. This order was determined by my own perceived importance of each predictor, adding the most impactful predictors first. 

- popularity: most direct measure of public engagement/attention, directly driving ticket sales/streaming
- genres: certain genres may systematically generate higher box office revenues (e.g. action movies versus documentaries)
- release_month: certain holidays/seasons may be an insightful factor to box office performance, making it a meaningful profit driver
- runtime: investigating shorter films vs. longer films may provide useful information as to what audiences prefer to watch
- original_language: English films have a broader global distribution reach, but this partially overlaps with what genre may already capture
- adult: These films have a restricted audience, which already limits the revenue ceiling, but most films in the dataset are non-adult so its contribution likely to be smaller.

The models with these predictors have the following training and test performance metrics:

In [ ]:
# Adding the remaining predictors in the linear/unregularized model

genre_cols_clean = [col.replace(' ', '_') for col in genre_cols] # Replace spaces with underscores within genres to prevent errors in model formulas
lang_cols = [col for col in X_train.columns if col.startswith('original_language_')] # Grabbing all ohe language columns (instead of manually listing)
month_cols = [col for col in X_train.columns if col.startswith('release_month_')] # Grabbing all ohe month columns (instead of manually listing)

# Rename columns with spaces in X_train and X_test: leaving spaces causes KeyErrors
X_train = X_train.rename(columns = {'Science Fiction': 'Science_Fiction', 'TV Movie': 'TV_Movie'})
X_test = X_test.rename(columns = {'Science Fiction': 'Science_Fiction', 'TV Movie': 'TV_Movie'})



# Ordered dictionary where each key is a label for the table and each value is a cumulative list of predictors at each step
    # Each step builds on the previous one
predictor_steps = {
    'popularity': ['popularity'],
    'popularity, genres': ['popularity'] + genre_cols_clean,
    'popularity, genres, release_month': ['popularity'] + genre_cols_clean + month_cols,
    'popularity, genres, release_month, runtime': ['popularity'] + genre_cols_clean + month_cols + ['runtime'],
    'popularity, genres, release_month, runtime, original_language': ['popularity'] + genre_cols_clean + month_cols + ['runtime'] + lang_cols,
    'popularity, genres, release_month, runtime, original_language, adult': ['popularity'] + genre_cols_clean + month_cols + ['runtime'] + lang_cols + ['adult']
}

results = []

# Looping through each entry in predictor_steps, creating a linear unregularized model on the current set of predictors 
    # and adding training/test performance metrics for comparison
for label, predictors in predictor_steps.items():
    model = LinearRegression()
    model.fit(X_train[predictors], y_train)
    
    y_pred_train = model.predict(X_train[predictors])
    y_pred_test = model.predict(X_test[predictors])
    
    results.append({
        'Predictors Added': label,
        'Training RMSE': round(root_mean_squared_error(y_train, y_pred_train), 2),
        'Test RMSE': round(root_mean_squared_error(y_test, y_pred_test), 2),
        'Training MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
        'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2),
        'Training R²': round(model.score(X_train[predictors], y_train), 4),
        'Test R²': round(model.score(X_test[predictors], y_test), 4)})

pd.DataFrame(results).style.hide(axis = 'index').set_properties(subset = ['Predictors Added'], **{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]},
    {'selector': 'th.col0', 'props': [('text-align', 'left')]}])

Predictors Added,Training RMSE,Test RMSE,Training MAE,Test MAE,Training R²,Test R²
popularity,85513064.020000,74852384.120000,24842043.400000,23697003.660000,0.057200,0.005600
"popularity, genres",83845230.430000,73910802.290000,24558881.960000,23690161.340000,0.093600,0.030500
"popularity, genres, release_month",83685284.870000,73688898.140000,25512537.080000,24531277.790000,0.097000,0.036300
"popularity, genres, release_month, runtime",83121514.350000,73176915.760000,27182909.700000,26418285.870000,0.109200,0.049700
"popularity, genres, release_month, runtime, original_language",82724253.450000,72823171.050000,28036541.680000,27412113.450000,0.117700,0.058800
"popularity, genres, release_month, runtime, original_language, adult",82721748.810000,72818575.120000,28042236.000000,27414291.840000,0.117700,0.058900


**Results:** As the number of predictors increased, it the linear and unregularized model's performance also increased.


1. Training R² went from 0.05 to 0.117. Test R² also increased, going from 0.0056 to 0.058. The **increases in R²** indicate that with more predictors added, the model is able to explain more variance in the response (profit).
2. Training **RMSE decreased** from 85.5M to 82.7M dollars, while Test **RMSE decreased** from 74.8M dollars to 72.8M dollars. This suggests that the model's worst-case errors are gradually shrinking.
3. Training **MAE** goes from 24.8M to 28M dollars, while Test **MAE** goes from 23.6M to 27.4M dollars. Adding genre and language dummy variables may have shifted model predictions in ways that reduce large errors (improving RMSE and R²), but increase the average error across typical films (worsening MAE). This is important to consider because it indicates that the model gets better at handling extreme blockbuster movies, but slightly worse in typical range films, suggesting that this dataset is heavily influenced by outliers that a linear, unregularized model struggles to handle alongside typical films.

#### Non-linear/Regularized Model

As for the nonlinear, unregularized model, I decided to investigate the **2nd order terms** of the predictors, as well as the interaction terms. I chose to utilize **Lasso** regression with Python's LassoCV object, since it can be especially helpful when picking useful non-linearities. When all interaction and 2nd degree polynomial terms are generated, there will be a very large number of new features. **Lasso prioritizes performing predictor selection** by shrinking coefficients that don't provide much revelant information to 0, so the predictors that are retained are those that have a bigger influence on profit. Since LassoCV automatically generates its own array based on the data, I chose to not manually define my own array of alphas to find the best tuned hyperparameter, setting the max_iter input to 10,000 (also because of computational limitations). It is also important to acknowledge that this decision means I had less control over the verification of the best alpha value, but also avoids risks of searching inappropriate ranges.

As for Ridge regression, it shrinks all coefficients toward 0 but never exactly to 0, meaning every polynomial term retains some influence regardless of how relevant it is in this task. For this reason, I didn't decide to use Ridge regression.

**Training the Model:**

I first scaled the train and test data using scikit-learn's StandardScaler object. I then created the 2nd order polynomial and interaction terms with scikit-learn's PolynomialFeatures object. After both of these steps, I trained the model using scikit-learn's Lasso cross validation object (LassoCV) to find the "tuned" hyperparameter:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Lasso, LassoCV

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Degree 2 polynomials and interaction terms
poly = PolynomialFeatures(degree = 2, include_bias = False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)


# LassoCV to find best alpha
lcv = LassoCV(cv = 5, random_state = 16, max_iter = 10000)
lcv.fit(X_train_poly, y_train)

print(f"Best alpha: {lcv.alpha_}")

Best alpha: 3429325.392696592


The best alpha of 3,429,325 is very large, but it also indicates that Lasso applied **aggressive regularization** and **zeroed out** the vast majority of coefficients, keeping only the informative terms. This also provides us insights about noise present in movie profit. Generally, movie profit is a noisy variable dominated by unpredictable factors like audience reception or cultural timing that the predictors used may not fully capture. With so much unexplained variance, the cross validation process decided that most terms were fitting noise, so aggressive regularization was needed to prevent overfitting. Additionally, because of the large number of features after one-hot-encoding, the degree 2 polynomial expansion created hundreds of candidate terms. When the number of possible terms is large relative to predictive signals, a large alpha effectively distinguishes the few informative terms from the many uninformative ones. This information suggests that the terms that survived, including popularity or popularity x runtime (shown below), survived after extremely aggressive penalization that zeroed out a majority of the terms. Therefore, these surviving terms represent more meaningful predictors of movie profit than they would be with a lower alpha that permitted more terms to survive.

Next, before evaluating the trained and tuned lasso regression model on the training and test data, it may be important to investigate the coefficients of each term. In order to do so, since the purpose of Lasso regression is **predictor selection**, I filtered out terms with nonzero coefficients and ordered the rest of the coefficients by their magnitudes in descending order:

In [ ]:
coefficients[coefficients['Coefficient'] != 0].reindex(
    coefficients[coefficients['Coefficient'] != 0]['Coefficient'].abs().sort_values(ascending = False).index).style.hide(axis = 'index')

Term,Coefficient
popularity,23689514.984142
popularity runtime,16867826.488874
popularity Adventure,7743083.417891
runtime,6663376.673534
popularity original_language_en,5476992.948019
runtime Adventure,3881332.874327
runtime Science_Fiction,3578165.138943
popularity release_month_9,-3555560.096921
popularity release_month_12,3477134.061116
popularity Fantasy,3139990.490508


Looking at the table of all nonzero term and their coefficients, we are able to see the **3 most important** terms:

In [ ]:
poly_feature_names = poly.get_feature_names_out(X_train.columns) # Feature names

coefficients = pd.DataFrame({'Term': poly_feature_names, 'Coefficient': lcv.coef_}) # coefficients of each term

top_3 = coefficients[coefficients['Coefficient'] != 0]  # remove terms Lasso zeroed out
top_3 = top_3.reindex(top_3['Coefficient'].abs().sort_values(ascending = False).index)
top_3 = top_3.head(3).reset_index(drop = True)

top_3.style.hide(axis = 'index')

Term,Coefficient
popularity,23689514.984142
popularity runtime,16867826.488874
popularity Adventure,7743083.417891


After finding the top 3 most important terms, I then evaluated the trained and tuned lasso regression model onto the training and test data, obtaining the **training/test performance**:

In [ ]:
# Evaluate on test set
y_pred_test = lcv.predict(X_test_poly)
y_pred_train = lcv.predict(X_train_poly)

# Dataframe of training and test performances
pd.DataFrame([{
    'Training RMSE': round(root_mean_squared_error(y_train, y_pred_train), 2),
    'Test RMSE': round(root_mean_squared_error(y_test, y_pred_test), 2),
    'Training MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2),
    'Training R²': round(lcv.score(X_train_poly, y_train), 4),
    'Test R²': round(lcv.score(X_test_poly, y_test), 4)}]).style.hide(axis = 'index')

Training RMSE,Test RMSE,Training MAE,Test MAE,Training R²,Test R²
75450088.250000,69520429.820000,21311488.320000,20734819.930000,0.266000,0.142300


**Results:**

In the **linear model**, Test RMSE with the linear terms of the 6 predictors was 72.8M dollars, with a R² value of 0.059 (i.e the model explains 5.9% of the variance in predicted profit). In the **regularized, polynomial model** with respective 2nd degree terms, test RMSE decreased to 69.5M dollars and R² increased to 0.142 (i.e the model explains 14.2% of the variance in predicted profit). 

Together, test R² more than doubled, and test RMSE improved by a few million dollars, confirming that polynomial and interaction terms captured nonlinear signals that the nonlinear regression model failed to capture. In other words, the **regularized polynomial model meaningfully outperformed the baseline linear model.**

#### Interpretation

Comparing the results of the linear/unregularized model vs. the non-linear (polynomial)/regularized model, information regarding the extent of each predictors' influence and possible non-linearities.

**1. The most informative predictor(s)**:
    
`popularity` seems to be the overwhelmingly most important predictor. In the linear model alone, adding popularity alone as the first predictor achieved a test RMSE of 74.9M and an R² of 0.0056. Looking at the table of the terms' coefficients in the polynomial, regularized model, `popularity` alone is the highest coefficient (23.7M). The 2nd and 3rd highest coefficient is popularity's interaction with runtime (16.9M), and Adventure (~7.7M). Some other notable terms include popularity's interaction with original_language_en (5.5M) and release_month_9 (-3.6M). The fact that popularity primarily dominates the coefficient table and survived Lasso's penalization consistently confirms that it is one of, if not the single strongest driver of movie profit in this dataset. 

Another informative predictor is the movie's `runtime`. Again looking at the linear model, adding runtime improved R² from 0.036 to 0.05, and reduced test RMSE from 73.7M to 73.2M. In the nonlinear, regularized model runtime as the standalone coefficient is the 4th highest (6.7M), and it also appears in its interaction terms with Adventure (~3.9M), and Science Fiction (3.6M). also noting its interaction with popularity stated above. This suggests that runtime's effect on profit also depends on the genre. Longer films generally generate more profit, but particularly when they're adventure or science fiction films.

`genres` collectively represents a major source of predictive signal. In the linear model, adding genres produced the largest incremental improvement, with test R² jumping from 0.006 to 0.031 and test RMSE dropping from 74.9M to 73.9M, a difference larger than any subsequent predictor addition. In the nonlinear regularized model, `Adventure` appears in terms with high coefficients, particularly in its interaction with popularity and runtime, and is generally present in other surviving terms after Lasso regularization (e.g. Adventure x original_language_en, Adventure²), indicating that it is a very influential individual genre.

Ultimately, when comparing the training and test performance of the overall linear model with all 5 predictors versus the nonlinear, regularized model with all predictors and its 2nd degree and/or interaction terms, the overall model performance improved substantially to a Test R² of 0.143 and Test RMSE of $69.5M.

**2. The least informative predictor(s)**:

`adult` is the least informative predictor by every available measure. In the linear model, adding adult after all the predictors produced minimal change, test R² increasing from 0.0588 to 0.0589 and test RMSE barely decreasing from 72,823,171 to 72,818,575. Additionally, when looking at the coefficients of the nonlinear regularized model, adult does not appear in a single surviving term after Lasso regularization, indicating that terms with adult were driven to equal 0, meaning that the information it provided was not relevant in predicting profit.

`original_language` is also relatively uninformative as a standalone predictor. In the linear model, adding it only improved test R² from 0.0497 to 0.0588 and reduced test RMSE by about 354K, also representing minimal meaningful improvement when compared to other predictor groups. In the nonlinear regularized model, most surviving language terms are squared terms with relatively small coefficients (e.g. original_language_other² at about -757K) or interacting with certain movie genres (e.g. War original_language_zh at about 560K). There is an exception of popularity x original_language_en  (about $5.5M), so it definitely still provides more information than adult, but not as much as predictors like popularity, runtime, or individual genres.

**3. Non-linearities in the underlying relationship**:

The results provide strong and consistent **evidence of nonlinearities in the underlying relationship** with profit. When looking at the linear model's test performance, RMSE was 72.8M dollars, MAE was 27.4M dollars, and R² 0.0589. On the other hand, the nonlinear regularized model had a test RMSE of 69.5M, MAE of 20.7M, and R² of 0.1423. With this information, it is clear that every evaluation metric improved when polynomial and interaction terms were introduced. Test R² more than doubles from 0.0589 to 0.1423, RMSE improved by about 3.3M, and MAE improved by about 6.7M, suggesting that the nonlinear model is not just better at handling extreme blockbuster outliers, but is genuinely more accurate on typical films as well.

Examining the specific **surviving coefficients** further support the nature of these nonlinearities as well. Though popularity alone is the largest coefficient after regularization, the other terms with the highest coefficients are interaction terms (e.g.popularity x runtime with a coefficient of 16.9M or popularity x Adventure with a coefficient of 7.7M), exceeding the coefficients of other standalone predictors. This demonstrates that the combined effect of these variables on profit is substantially larger than what their individual linear contributions would suggest. If the relationship was purely linear, interaction terms would not have survived Lasso penalization.

Additionally, the **quadratic terms are present** for several genre variables (e.g. Adventure² with a coefficient of about 1.3M, Fantasy² with a coefficient of about 666K or Science_Fiction² with a coefficient of about 283K). Upon further inspection on the table, only these quadratic terms survived, while none of their respective linear terms did. This information suggest nonlinear returns in these genres, where the more a film leans more towards these genres, the more profit it disproportionately generates.

Lastly, **seasonal nonlinearities** are also evident through the release month interaction terms. popularity x release_month_12 (coefficient of about 3.47M) and popularity x release_month_9 (coefficient of about -3.55M) indicate that the effect of popularity on profit is not constant throughout the year but varies by release month. A popular film released in December generates substantially more profit than an equally popular film released in September. This type of conditional relationship, where one predictor depends on another, is inherently nonlinear and can not be represented by a standard linear model.

Conclusively, the robustness of these findings is further supported by the fact that all surviving nonlinear terms passed through aggressive Lasso penalization that zeroed out the vast majority of hundreds of polynomial terms. The terms that survived did so because their predictive power was strong enough to outweigh the regularization penalty.

## 6) Inference

The inference model was constructed with statsmodels based on the findings from the prediction section, where Lasso regularization with 2nd degree polynomial features identified the most informative terms. The base predictors `popularity`, `runtime`, `Adventure`, `Science_Fiction`, `original_language_en`, and `release_month_12` were included as linear terms to include original predictors and to provide interpretable baseline coefficients. Notably, genre variables such as Adventure and Science_Fiction did not appear as standalone terms in the Lasso model. They only survived as interaction terms with popularity and runtime. Regardless, they are included in the creation of the model as base predictors because interaction terms require their constituent variables to be present in the model for correct interpretation. The nonlinear terms, including popularity², runtime², and several interaction terms, were selected directly from the surviving Lasso coefficients from the prediction section.

After creating the model, we are able to see a comprehensive overview of regression model results:

In [ ]:
import statsmodels.formula.api as smf

# Replace spaces with underscores in columns to avoid errors in statsmodels formula
inference_df = X_train.copy()
inference_df['profit'] = y_train.values
inference_df = inference_df.rename(columns = {'Science Fiction': 'Science_Fiction', 'TV Movie': 'TV_Movie'})

# Using : instead of * for interaction terms to avoid redundantly adding predictors again
formula = '''profit ~ popularity + runtime + Adventure + Science_Fiction + original_language_en + release_month_12 + 
I(popularity**2) + I(runtime**2) + popularity:runtime + popularity:Adventure + runtime:Adventure + runtime:Science_Fiction + 
popularity:original_language_en + popularity:release_month_12'''

inference_model = smf.ols(formula = formula, data = inference_df).fit()
inference_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 profit   R-squared:                       0.252
Model:                            OLS   Adj. R-squared:                  0.252
Method:                 Least Squares   F-statistic:                     482.0
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:27:52   Log-Likelihood:            -3.9134e+05
No. Observations:               20000   AIC:                         7.827e+05
Df Residuals:                   19985   BIC:                         7.828e+05
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
===========================================================================================================
                                              coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                7.251e+06   4.04e+06      1.795      0.073   -6.65e+05    1.52e+07
original_language_en[T.True]            -1.578e+06   1.32e+06     -1.192      0.233   -4.17e+06    1.02e+06
release_month_12[T.True]                -1.627e+07   2.25e+06     -7.242      0.000   -2.07e+07   -1.19e+07
popularity                              -1.748e+06   1.32e+05    -13.279      0.000   -2.01e+06   -1.49e+06
popularity:original_language_en[T.True]  1.483e+06    9.5e+04     15.604      0.000     1.3e+06    1.67e+06
popularity:release_month_12[T.True]      2.384e+06    1.3e+05     18.364      0.000    2.13e+06    2.64e+06
runtime                                 -1.218e+05   6.89e+04     -1.768      0.077   -2.57e+05    1.32e+04
Adventure                               -4.971e+07   7.11e+06     -6.996      0.000   -6.36e+07   -3.58e+07
Science_Fiction                          -9.12e+07   8.13e+06    -11.213      0.000   -1.07e+08   -7.53e+07
I(popularity ** 2)                      -1082.8879     22.229    -48.716      0.000   -1126.458   -1039.318
I(runtime ** 2)                           635.3300    308.555      2.059      0.040      30.536    1240.124
popularity:runtime                       1.492e+04    929.399     16.050      0.000    1.31e+04    1.67e+04
popularity:Adventure                     1.121e+06   4.28e+04     26.175      0.000    1.04e+06     1.2e+06
runtime:Adventure                         6.41e+05   6.87e+04      9.336      0.000    5.06e+05    7.76e+05
runtime:Science_Fiction                  1.046e+06   8.23e+04     12.706      0.000    8.84e+05    1.21e+06
==============================================================================
Omnibus:                    36033.204   Durbin-Watson:                   1.994
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        122100807.340
Skew:                          12.768   Prob(JB):                         0.00
Kurtosis:                     384.928   Cond. No.                     8.77e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.77e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""


**From these results**, it is worth it to discuss the effect of each predictor on profit, this model's reliability, and the amount of variation in profit explained by this model.

**1: Effect of Each Predictor on Profit**:

`popularity` has a negative coefficient of -1.7M, which seems counterintuitive at first. However, this should not be interpreted in isolation because popularity also appears in many nonlinear (interaction and higher order) terms of this model, so the total effect on popularity on profit also depends on the values of runtime, original_language_en, release_month_12, and popularity itself simultaneously. The negative linear term is partially offset by the positive interaction terms discussed below.

`popularity²` has a coefficient of -1082, suggesting that the relationship between popularity and profit is concave at high popularity levels, meaning extremely popular films do not generate proportionally more profit indefinitely. In other words, the marginal effect of popularity on profit decreases as popularity increases, where each additional unit of popularity generates less additional profit at higher popularity levels than at lower ones, suggesting diminishing returns for extremely popular films.

`popularity x runtime` has a coefficient of 14,920, meaning that for each additional minute of runtime, the effect of popularity on profit increases by $14,920. Simply put, popular films that are also longer generate disproportionately more profit.

`popularity x Adventure` has a coefficient of 1,121,000, meaning the effect of popularity on profit is amplified by approximately $1,121,000 for Adventure films compared to non-Adventure films. This is one of the strongest interaction effects in the model and supports the idea that popular Adventure films generate high profit relative to equally popular films in other genres.

`popularity x original_language_en` has a coefficient of 1,483,000, indicating that the effect of popularity on profit is approximately $1,483,000 higher for English language films than for non-English language films. This likely reflects the broader global distribution reach of English language productions, which allows popular English language films to monetize their popularity more effectively.

`popularity x release_month_12` has a coefficient of 2,384,000, meaning  films released in December generate approximately $2,384,000 more profit per unit of popularity than films released in other months. 

`runtime` has a negative coefficient of -121,800, suggesting a negative baseline relationship between runtime and profit for non-Adventure and non-Science Fiction films. In other words, the linear runtime coefficient reflects the effect of runtime when both Adventure and Science_Fiction equal zero. This negative baseline suggests longer films in other genres do not inherently generate more profit, though this effect is not statistically significant as its p-value = 0.077, which is above the conventional level of 0.05.

`runtime²` has a coefficient of 635.3, indicating a slight convex relationship between runtime and profit at the baseline level. In other words, at very long runtimes, profit begins to increase modestly for non-Adventure, non-Science Fiction films. However the small magnitude relative to other terms suggests this effect is limited.

`runtime x Adventure` has a coefficient of 641,000, meaning each additional minute of runtime increases profit by an additional $641,000 for Adventure films specifically. The longer the film under the genre of Adventure, the more profit it is expected to generate.

`runtime x Science_Fiction` has a coefficient of 1,046,000, meaning each additional minute of runtime increases profit by an additional $1,046,000 for Science Fiction films specifically. This value is larger than runtime's interaction with Adventure suggesting that runtime has a stronger effect on Science Fiction films than for Adventure films.

`Adventure` has a coefficient of -49,710,000, representing the baseline profit for Adventure films when runtime and popularity are zero. Since Adventure appears in interaction terms with both popularity and runtime, this large negative baseline is not meaningful on its own, as it rather represents a hypothetical situation of an Adventure film of 0 minutes of runtime and 0 popularity.

`Science_Fiction` has a coefficient of -91,200,000, similarly representing the baseline profit for Science Fiction films when runtime is zero. This large negative baseline is offset as runtime increases through the runtime x Science_Fiction interaction term of $1,046,000 per minute, meaning longer Science Fiction films progressively recover and exceed this baseline. 

`original_language_en` has a coefficient of -1,578,000, representing the baseline profit difference for English language films when popularity is zero. This is not statistically significant and should not be interpreted independently, given its p-value of 0.233. The meaningful contribution of original_language_en comes entirely through the popularity × original_language_en interaction term.

`release_month_12` has a coefficient of -16,270,000, representing the baseline profit for December releases when popularity is zero. The negative baseline reflects a low-popularity scenario and the meaningful contribution of release_month_12 comes through its interaction with popularity. Put simply, popular films released in December generate substantially more profit than equally popular films released in other months.

**2: Reliability of the Effect of the Predictor**:

The vast majority of the predictors are statistically significant. The predictors `popularity`, `popularity²`, `popularity x runtime`, `popularity x Adventure`, `popularity x original_language_en`, `popularity x release_month_12`, `Adventure`, `Science_Fiction`, `runtime x Adventure`, `runtime x Science_Fiction`, `release_month_12` all have a p-value of 0, which is less than the standard 0.05 threshold for statistical significance.

Additionally, `runtime` has a p-value of 0.04 while `runtime²` has a p-value of 0.077. This suggests that runtime's quadratic relationship with profit is not statistically significant. Though the linear term runtime is statistically significant with profit, it may be more insightful to examine its interaction terms, especially with Adventure and Science Fiction films.

Lastly, with a p-value of 0.233, `original_language_en` is not statistically significant, so its effect on the response is not as reliable compared to the predictors stated above.


Ultimately, when considering the whole picture, `popularity` has the most **reliable and well-established** relationship with profit across all of its terms. Its linear, quadratic, and interaction terms with runtime, Adventure, original_language_en (p-values = 0) are all highly statistically significant. As for `runtime`, it has **moderately reliable** relationship with profit. The standalone linear runtime term is not statistically significant (p-value = 0.07 > 0.05), means that there is insufficient evidence to conclude that runtime has a reliable effect on profit for non-Adventure and non-Science Fiction films on its own. However, the quadratic term and interaction terms with Adventure and Science Fiction films are statistically significant, suggesting that runtime's relationship with profit is not simply linear, but interwoven with  the film's genre. Finally, `original_language_en` has an **unreliable effect** on its own, but a **reliable** interaction effect. The standalone p-value is 0.233, which is not statistically significant. However, as said above, its interaction term with popularity is highly significant, indicating that a film in English by itself does not reliably predict profit, but is amplified with its interaction with that film's popularity level.



Overall, the pattern of statistical significance across the model is coherent. The predictors that showed the large coefficients in the Lasso regularized model from the prediction section are also the ones with the most consistently significant terms.

**3: Amount of Variance in Profit the Model Captures**:

The model achieves an R² of **0.252**, meaning that it explains approximately **25.2%** of the variation in profit across the training data. While 25.2% may seem quite minimal, it represents a meaningful improvement over the full linear model's training R² of 0.1177 and test R² of 0.058. It is also consistent with the implicit difficulty of predicting movie profit. In my model, I only used a small handful of predictors to predict movie profit, while in reality, profit is influenced by a great multitude of factors, even those not included in this dataset (e.g. marketing spend, cast members, distribution channels). Though this analysis was conducted on a limited set of predictors, there are still many meaningful insights that can be extracted from the findings.

## 7) Recommendation to Stakeholders

#### Main Action Items for Stakeholders

Based on the information and insights given from these models, there are some recommendations I have for movie production companies to take when trying to maximize profit.

**Prioritize popularity**: From prior analyses, popularity appears to be a consistent, strong indicator of profit. Production companies and distributors should invest in efficient marketing campaigns, social media presence, advertisements, and build up hype before release so that the film's popularity score is high at the time of release, generating as much excitement among the public as possible to maximize expected generation of profit.

**Prioritize Adventure and Science Fiction Films, with longer runtimes**: The interaction between runtime and Adventure/Science Fiction genres is a very reliable finding in this model. Production companies should be willing to invest in longer films of Adventure and Science Fiction films, rather than shortening them (for commercial reasons, for example), as each additional minute of films in these genres are expected to generate more profit.

**Target December releases for high-popularity films specifically**: The interaction between popularity and release_month_12 (films released in December) is about $2,384,000 per unit of popularity, which especially benefits popular films. Therefore, production companies and marketing teams should first identify their most anticipated, high-visibility productions and prioritize releasing them in the month of December. As a result, it is appropriate to be more flexible with the release date of films with less public anticipation/visbility, where the benefit is less pronounced.

#### Limitations of this Analysis

**Severely skewed profit distribution**: Despite filtering observations where revenue and budget was 0, profit remains heavily right-skewed with a small handful of blockbusters generating hundreds of millions, or even billions while most films have a profit of much less. The decision to retain blockbuster outliers was because these films are not necessarily an error or anomaly in the data, but real observations that represent real financial outcomes. Removing these outliers would mean deliberately excluding the most successful films from an analysis that specifically aims to predict movie profit. That said, retaining the outliers did came at a statistical cost, including having an effect on model evaluation metrics.

**Number of predictors in the data**: The model only explains 25.2% of the variation in profit, meaning that about 74.8% of profit variance is captured by factors not used as predictors or those not present in this dataset. Critical predictors that intuitively have a large influence on movie profit include cast/director star power, marketing spend, distribution channels, and more. As a result, the analyses performaed has been limited by not having access to additional important influences in movie profit.

**Random sampling**: Due to computational reasons, this analysis was conducted on a set of 25,000 randomly sampled observations from the original dataset with over 1 million films (before cleaning steps were performed). By random sampling, that this subset may not be fully representative of original dataset, where even high-revenue block busters are under represented due to these outlier films having significantly less observations than where a majority of the data clustered.

**Time Considerations**: This dataset spans from a long time period, including films starting from 1898 to present day. Considering this, the relationship between predictors and profit likely changed significantly over time due to economic inflation, evolution of film production, the rise of streaming services, changes in audience preferences, etc. This model does not account for evolving trends over time, and coefficients estimated using historical data may not be as generalizable in predicting profits in a present-day context.

**Popularity Score**: The popularity score in The Movie Database is derived from user engagement and ratings, which may themselves be influenced by a film's commercial success after the fact. In other words, it is possible for high-profit films to generate high popularity scores after release, rather than high popularity/anticipation before release being the main factor driving profit.

#### Addressing these Limitations

**Skewed distribution of profit**: A possible step to take to mitigate the extreme influence of blockbuster outliers is to apply a log transformation to profit to "compress" the profit ranges. In my analyses, I chose to keep profit in its original form, to help with easier interpretability. However, if a log transformation were added, this could have possibly helped profit become more normally distributed rather than extremely skewed. More generally, other regression techniques that are less sensitive to extreme observations and handles outliers better than what the techniques this analysis was conducted on would be helpful to consider.

**Limited number of predictors**: Since I only used the singular dataset on found on Kaggle, the most impactful improvement to make is to enrich this dataset with additional predictor variables from external sources. For example, looking at specific box office/industry databases like Motion Picture Association, Box Office Mojo, or Rotten Tomatoes can provide valuable information regarding spending/marketing budgets, cast/director star power, and critic evaluation metrics that will likely be able to explain a much higher percentage of variation in profit than the predictors utilized in the original TMDB dataset.

**Random sampling**: The most apparent approach to address the limitations of smaller sample sizes is to utilize the full dataset, or significantly increase the sample size so the data that analysis is conducted upon is more representative. Because of the limitations of computational resources and time, this may not always be possible. Therefore, another possibility is to repeat the analysis across different set seeds. The data was derived by setting a seed. If this process was repeated across different samples and set seeds, I can examine whether coefficients and performance metrics remain stable. If so, this would confirm that the original sample is sufficiently representative.

**Time Considerations**: It would be insightful to also include the release year as a predictor in the model, instead of only extracting the month. When considering the year's polynomial terms, it can provide more insights about nonlinear trends over time (e.g. rapid growth of box office revenues and effects of streaming services beginning in the 2010s). Additionally, adjusting all financial figures for inflation using the Consumer Price Index (CPI) before modeling can also be useful to ensure that financial results are more comparable.

**Popularity score**: If possible, a way to address post-release popularity scores is to replace it with pre-released popularity scores that are available before a film's scheduled release. This information can be found by investigating social media mention counts and sentiment scores leading up to the film's release, trailer view counts on YouTube, serach trend data on Google Trends, and more.

## 8) Conclusion

The purpose of this project and analysis was to predict movie profits and understand the role of certain factors in driving movie profit. Across both prediction and inference analyses, popularity emerged as the dominant factor in influencing movie profit, consistently producing the largest coefficients in regularized polynomial models and meaningful model performances. Additionally, while constructing a linear model and adding one predictor at a time, it was found that genre (particularly Adventure and Science Fiction) and runtime also had meaningful contributions in predicting profit beyond popularity alone. On the other hand, adult films and the original language of the film was quite uninformative as standalone predictors. 

Transitioning from an unregularized linear model, which explained about 5.9% of the variance in profit, to a Lasso regularized polynomial model, which explained about 14.3% of the variance in profit, supports the idea that there are nonlinearities in the underlying relationship between the predictors and movie profit. Particularly, the interaction with popularity and adventure films, as well as runtime's dependency on adventure and science fiction genres was of the most notable. Inference analyses of evaluating the statistically significant coefficients across nearly all key terms further revealed the conditional nature of the English language and releasing films in December, both of which highly depend on film popularity. It is true that the models created explains a relatively small percentage of variation in profit, highlighting the difficulty of predicted profit in reality, especially in the absence of other key predictors such as marketing budget/spending and cast/director influences. Regardless, the findings still provide actionable and statistically reliable guidance for film production companies, studios and marketing teams seeking to maximize the profit potential of their productions.

## References

[1] TheMovieDatabase. Popularity & Trending, 2025. https://developer.themoviedb.org/docs/popularity-and-trending